# 🚨 Analyse Délinquance France — Niveau Département
**Source** : `crimes-et-delits-enregistres-par-les-services-de-gendarmerie-et-de-police-depuis-2012.xlsx`  
**Période** : 2012 → 2021  
**Services** : Police Nationale (PN) + Gendarmerie Nationale (GN)  
**Granularité** : 105 départements × 107 index de crimes

### Structure du fichier source
- 20 onglets : `Services PN 2012…2021` + `Services GN 2012…2021`
- **PN** : 3 lignes d'en-tête (ligne 0 = département, ligne 1 = périmètre, ligne 2 = libellé zone)
- **GN** : 2 lignes d'en-tête (ligne 0 = département, ligne 1 = libellé zone)
- Données : une colonne par zone (circumscription/compagnie), agrégées au département

### Plan
1. Imports  
2. Parsing & construction du DataFrame département  
3. EDA univariée  
4. Analyse départementale  
5. Comparaison PN vs GN  
6. Modélisation Prophet  
7. XGBoost + LightGBM  
8. Rapport final

## 1. Imports

In [ ]:
import subprocess, sys
for pkg in ['pandas', 'openpyxl', 'matplotlib', 'seaborn',
            'prophet', 'xgboost', 'lightgbm', 'scikit-learn']:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])
print('Librairies OK')

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

warnings.filterwarnings('ignore')
plt.style.use('default')
sns.set_palette('husl')
print('Imports OK')

## 2. Parsing & Construction du DataFrame

Le fichier contient les données par **zone** (circumscription PN, compagnie GN).  
On agrège au **département** en sommant toutes les zones d'un même département.

In [ ]:
# Adapter ce chemin si le fichier est ailleurs
FICHIER = 'crimes-et-delits-enregistres-par-les-services-de-gendarmerie-et-de-police-depuis-2012.xlsx'

import warnings
warnings.filterwarnings('ignore', category=pd.errors.PerformanceWarning)

xl = pd.ExcelFile(FICHIER)
onglets = [s for s in xl.sheet_names if s != 'Présentation']
print(f'{len(onglets)} onglets de données :')
print(onglets)

In [ ]:
def parse_pn(xl, onglet):
    """
    Parse un onglet 'Services PN AAAA'.
    Structure :
      ligne 0 : titre | 'Départements' | dept dept dept ...
      ligne 1 : NaN   | 'Périmètres'   | DCSP DCSP DCPAF ...
      ligne 2 : 'Code index' | 'Libellé' | zone zone zone ...
      lignes 3+ : code_crime | libelle | valeur valeur ...
    """
    df = pd.read_excel(xl, sheet_name=onglet, header=None).copy()
    annee = int(str(df.iloc[0, 0]).split()[1])

    depts        = df.iloc[0, 2:].values          # département par zone
    codes_crime  = df.iloc[3:, 0].values           # index 1..107
    libelles     = df.iloc[3:, 1].values           # libellé crime
    valeurs      = df.iloc[3:, 2:].apply(pd.to_numeric, errors='coerce').values

    records = []
    for j, dept in enumerate(depts):
        dept_str = str(dept).strip().zfill(2) if str(dept) not in ['nan', 'None'] else None
        if dept_str is None:
            continue
        for i in range(len(codes_crime)):
            records.append({
                'annee': annee, 'service': 'PN',
                'departement': dept_str,
                'code_index': codes_crime[i],
                'libelle_crime': libelles[i],
                'valeur': valeurs[i, j]
            })
    return pd.DataFrame(records)


def parse_gn(xl, onglet):
    """
    Parse un onglet 'Services GN AAAA'.
    Structure :
      ligne 0 : titre | 'Départements' | dept dept dept ...
      ligne 1 : 'Code index' | 'Libellé' | zone zone zone ...
      lignes 2+ : code_crime | libelle | valeur valeur ...
    """
    df = pd.read_excel(xl, sheet_name=onglet, header=None).copy()
    annee = int(str(df.iloc[0, 0]).split()[1])

    depts        = df.iloc[0, 2:].values
    codes_crime  = df.iloc[2:, 0].values
    libelles     = df.iloc[2:, 1].values
    valeurs      = df.iloc[2:, 2:].apply(pd.to_numeric, errors='coerce').values

    records = []
    for j, dept in enumerate(depts):
        dept_str = str(dept).strip().zfill(2) if str(dept) not in ['nan', 'None'] else None
        if dept_str is None:
            continue
        for i in range(len(codes_crime)):
            records.append({
                'annee': annee, 'service': 'GN',
                'departement': dept_str,
                'code_index': codes_crime[i],
                'libelle_crime': libelles[i],
                'valeur': valeurs[i, j]
            })
    return pd.DataFrame(records)


print('Parsing en cours...')
dfs = []
for onglet in onglets:
    if 'PN' in onglet:
        dfs.append(parse_pn(xl, onglet))
    else:
        dfs.append(parse_gn(xl, onglet))
    print(f'  OK {onglet}')

df_raw = pd.concat(dfs, ignore_index=True)
print(f'\ndf_raw : {df_raw.shape}')
df_raw.head()

In [ ]:
# --- Nettoyage ---
df_raw['valeur'] = pd.to_numeric(df_raw['valeur'], errors='coerce')
df_raw = df_raw.dropna(subset=['valeur'])

# Normalisation code département (2A/2B conservés, autres sur 2 chiffres)
def norm_dept(d):
    d = str(d).strip().upper()
    if d in ('2A', '2B', '971', '972', '973', '974', '975', '976'):
        return d
    try:
        return str(int(float(d))).zfill(2)
    except:
        return d

df_raw['departement'] = df_raw['departement'].apply(norm_dept)

# --- Agrégation au département (somme toutes zones) ---
# Niveau 1 : par service (PN ou GN)
df_dep_svc = (
    df_raw
    .groupby(['annee', 'service', 'departement', 'code_index', 'libelle_crime'])
    ['valeur'].sum()
    .reset_index()
    .rename(columns={'valeur': 'nb_faits'})
)

# Niveau 2 : PN + GN combinés (total département)
df_dep = (
    df_raw
    .groupby(['annee', 'departement', 'code_index', 'libelle_crime'])
    ['valeur'].sum()
    .reset_index()
    .rename(columns={'valeur': 'nb_faits'})
)

print(f'df_dep (PN+GN agrégés) : {df_dep.shape}')
print(f'  Années       : {sorted(df_dep["annee"].unique())}')
print(f'  Départements : {df_dep["departement"].nunique()}')
print(f'  Crimes (index): {df_dep["code_index"].nunique()}')
df_dep.head()

In [ ]:
# --- Top 15 crimes par volume total ---
top_crimes = (
    df_dep.groupby(['code_index', 'libelle_crime'])['nb_faits']
    .sum().sort_values(ascending=False).head(15)
)
print('Top 15 crimes par volume (2012-2021) :')
for (code, lib), val in top_crimes.items():
    print(f'  [{int(code):3d}] {lib[:55]:<55} {val:>10,.0f}')

## 3. EDA Univariée

In [ ]:
# --- Évolution nationale totale ---
evol_nat = df_dep.groupby('annee')['nb_faits'].sum().reset_index()

plt.figure(figsize=(12, 5))
plt.bar(evol_nat['annee'], evol_nat['nb_faits'] / 1e6,
        color='steelblue', alpha=0.85, edgecolor='white')
plt.plot(evol_nat['annee'], evol_nat['nb_faits'] / 1e6,
         'o-', color='darkred', lw=2, ms=6)
plt.title('Total faits enregistrés en France (PN + GN)', fontsize=15, fontweight='bold')
plt.xlabel('Année'); plt.ylabel('Millions de faits')
plt.xticks(evol_nat['annee'])
plt.grid(True, alpha=0.3, axis='y'); plt.tight_layout(); plt.show()

In [ ]:
# --- Évolution des top 6 crimes ---
top6_codes = top_crimes.index.get_level_values('code_index')[:6]

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for i, code in enumerate(top6_codes):
    sub = df_dep[df_dep['code_index'] == code]
    libelle = sub['libelle_crime'].iloc[0][:40]
    evol = sub.groupby('annee')['nb_faits'].sum()
    axes[i].plot(evol.index, evol.values, 'o-', lw=2.5, ms=7, color='steelblue')
    axes[i].fill_between(evol.index, evol.values, alpha=0.15, color='steelblue')
    axes[i].set_title(f'[{int(code)}] {libelle}', fontweight='bold', fontsize=9)
    axes[i].set_xlabel('Année'); axes[i].set_ylabel('Nb faits')
    axes[i].grid(True, alpha=0.3)

plt.suptitle('Évolution des 6 crimes les plus fréquents (France)', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# --- Part PN vs GN ---
svc_total = df_dep_svc.groupby('service')['nb_faits'].sum()
plt.figure(figsize=(7, 5))
plt.pie(svc_total.values, labels=svc_total.index,
        autopct='%1.1f%%', colors=['steelblue', 'coral'],
        startangle=90, wedgeprops={'edgecolor': 'white', 'linewidth': 2})
plt.title('Répartition des faits : Police (PN) vs Gendarmerie (GN)', fontweight='bold')
plt.tight_layout(); plt.show()

## 4. Analyse Départementale

In [ ]:
# --- Top 20 départements (volume total 2012-2021) ---
top20_dep = (
    df_dep.groupby('departement')['nb_faits']
    .sum().sort_values(ascending=False).head(20)
)

q3 = top20_dep.quantile(0.75); med = top20_dep.median()
colors = ['#d73027' if v > q3 else '#fc8d59' if v > med else '#fee090'
          for v in top20_dep.values]

plt.figure(figsize=(12, 8))
top20_dep.plot(kind='barh', color=colors, alpha=0.9)
plt.title('Top 20 départements — volume total de faits (2012-2021)',
          fontweight='bold', fontsize=14)
plt.xlabel('Nombre total de faits')
plt.grid(True, alpha=0.3, axis='x'); plt.tight_layout(); plt.show()

print('Top 10 :')
for dept, val in top20_dep.head(10).items():
    print(f'  Dept {dept} : {val:,.0f} faits')

In [ ]:
# --- Heatmap Département × Crime (top 10 crimes) ---
top10_codes = top_crimes.index.get_level_values('code_index')[:10]
top10_libs  = [f"[{int(c)}] {l[:30]}" for c, l in top_crimes.head(10).index]

pivot_heat = (
    df_dep[df_dep['code_index'].isin(top10_codes)]
    .groupby(['departement', 'code_index'])['nb_faits']
    .sum().unstack('code_index').fillna(0)
)
pivot_heat.columns = top10_libs

# Normaliser par ligne (pour comparer les profils indépendamment du volume)
pivot_norm = pivot_heat.div(pivot_heat.sum(axis=1), axis=0)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(22, 20))

sns.heatmap(pivot_heat, ax=ax1, cmap='YlOrRd', annot=False,
            cbar_kws={'label': 'Nb faits (total 2012-2021)'})
ax1.set_title('Volume brut', fontweight='bold', fontsize=12)
ax1.set_xlabel(''); ax1.set_ylabel('Département')

sns.heatmap(pivot_norm, ax=ax2, cmap='RdYlGn_r', annot=False, fmt='.2f',
            cbar_kws={'label': 'Part relative par département'})
ax2.set_title('Profil relatif (part de chaque crime)', fontweight='bold', fontsize=12)
ax2.set_xlabel(''); ax2.set_ylabel('')

plt.suptitle('Heatmap Département × Crime', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('heatmap_dept_crimes.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# --- Évolution spaghetti par département (crime le plus fréquent) ---
crime_top1_code = top_crimes.index.get_level_values('code_index')[0]
crime_top1_lib  = top_crimes.index.get_level_values('libelle_crime')[0]

sub = df_dep[df_dep['code_index'] == crime_top1_code]
evol_dep = sub.pivot_table(values='nb_faits', index='annee', columns='departement', aggfunc='sum')

plt.figure(figsize=(14, 7))
for dep in evol_dep.columns:
    plt.plot(evol_dep.index, evol_dep[dep], color='gray', alpha=0.2, lw=0.8)
# Mettre en avant les top 5 départements
top5_deps = top20_dep.head(5).index
palette   = ['#d73027', '#fc8d59', '#fdae61', '#abd9e9', '#2c7bb6']
for dep, color in zip(top5_deps, palette):
    if dep in evol_dep.columns:
        plt.plot(evol_dep.index, evol_dep[dep], lw=3, color=color, label=f'Dept {dep}')
# Moyenne nationale
moy = sub.groupby('annee')['nb_faits'].sum()
plt.plot(moy.index, moy.values / evol_dep.shape[1], lw=3,
         color='black', ls='--', label='Moy. nationale')

plt.title(f'[{int(crime_top1_code)}] {crime_top1_lib[:60]}\nÉvolution par département',
          fontweight='bold')
plt.xlabel('Année'); plt.ylabel('Nb faits')
plt.legend(loc='upper right'); plt.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
# --- PCA des profils départementaux ---
pivot_pca = (
    df_dep.groupby(['departement', 'libelle_crime'])['nb_faits']
    .sum().unstack('libelle_crime').fillna(0)
)

X_pca  = StandardScaler().fit_transform(pivot_pca.values)
pca    = PCA(n_components=2)
coords = pca.fit_transform(X_pca)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))

ax1.scatter(coords[:, 0], coords[:, 1], s=80, alpha=0.8,
            c=range(len(pivot_pca)), cmap='tab20')
for i, dep in enumerate(pivot_pca.index):
    ax1.annotate(dep, (coords[i, 0], coords[i, 1]), fontsize=7,
                 ha='center', va='bottom')
ax1.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%})')
ax1.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%})')
ax1.set_title(f'PCA départements (variance: {pca.explained_variance_ratio_.sum():.1%})',
              fontweight='bold')
ax1.grid(True, alpha=0.3)

loadings = pd.DataFrame(pca.components_.T, columns=['PC1','PC2'], index=pivot_pca.columns)
top_l = loadings['PC1'].abs().sort_values(ascending=False).head(10)
ax2.barh(range(len(top_l)), top_l.values, color='steelblue', alpha=0.8)
ax2.set_yticks(range(len(top_l)))
ax2.set_yticklabels([x[:35] for x in top_l.index], fontsize=8)
ax2.set_xlabel('Contribution |PC1|'); ax2.set_title('Top contributeurs PC1', fontweight='bold')
plt.tight_layout()
plt.savefig('pca_departements.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Comparaison PN vs GN

In [ ]:
# Évolution PN vs GN dans le temps
evol_svc = (
    df_dep_svc.groupby(['annee', 'service'])['nb_faits']
    .sum().unstack('service')
)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Volume brut
evol_svc.plot(ax=ax1, marker='o', lw=2.5,
              color={'PN': 'steelblue', 'GN': 'coral'})
ax1.set_title('Évolution PN vs GN — volume brut', fontweight='bold')
ax1.set_xlabel('Année'); ax1.set_ylabel('Nb faits')
ax1.grid(True, alpha=0.3)

# Part relative
evol_pct = evol_svc.div(evol_svc.sum(axis=1), axis=0) * 100
evol_pct.plot(kind='bar', stacked=True, ax=ax2,
              color=['steelblue', 'coral'], alpha=0.85)
ax2.set_title('Part relative PN vs GN (%)', fontweight='bold')
ax2.set_xlabel('Année'); ax2.set_ylabel('%')
ax2.set_xticklabels(evol_pct.index, rotation=45)
ax2.legend(loc='upper right')

plt.tight_layout(); plt.show()

In [ ]:
# Crimes où GN domine vs crimes où PN domine
ratio_pn_gn = (
    df_dep_svc.groupby(['service', 'libelle_crime'])['nb_faits']
    .sum().unstack('service').fillna(0)
)
ratio_pn_gn['ratio_PN_GN'] = ratio_pn_gn['PN'] / (ratio_pn_gn['GN'] + 1)

top_pn = ratio_pn_gn.nlargest(10, 'ratio_PN_GN')
top_gn = ratio_pn_gn.nsmallest(10, 'ratio_PN_GN')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))

top_pn['ratio_PN_GN'].plot(kind='barh', ax=ax1, color='steelblue', alpha=0.85)
ax1.set_title('Crimes dominés par la Police (PN)', fontweight='bold')
ax1.set_xlabel('Ratio PN/GN'); ax1.set_yticklabels([x[:35] for x in top_pn.index], fontsize=8)
ax1.grid(True, alpha=0.3, axis='x')

top_gn['ratio_PN_GN'].plot(kind='barh', ax=ax2, color='coral', alpha=0.85)
ax2.set_title('Crimes dominés par la Gendarmerie (GN)', fontweight='bold')
ax2.set_xlabel('Ratio PN/GN (faible = GN domine)'); ax2.set_yticklabels([x[:35] for x in top_gn.index], fontsize=8)
ax2.grid(True, alpha=0.3, axis='x')

plt.tight_layout(); plt.show()

## 6. Modélisation Prophet

In [ ]:
from prophet import Prophet

TOP_N = 5
top_indicateurs = (
    df_dep.groupby(['code_index', 'libelle_crime'])['nb_faits']
    .sum().sort_values(ascending=False)
    .head(TOP_N).index.get_level_values('libelle_crime').tolist()
)
print('Crimes modélisés :', top_indicateurs)

def prep_prophet(df, libelle):
    """Série temporelle nationale (somme tous départements) pour Prophet."""
    return (
        df[df['libelle_crime'] == libelle]
        .groupby('annee')['nb_faits'].sum()
        .reset_index()
        .rename(columns={'annee': 'ds', 'nb_faits': 'y'})
        .assign(ds=lambda x: pd.to_datetime(x['ds'], format='%Y'))
    )

def rmse_safe(y_true, y_pred):
    mask = ~(np.isnan(y_true) | np.isnan(y_pred))
    return np.sqrt(np.mean((y_true[mask] - y_pred[mask])**2)) if mask.sum() >= 2 else np.nan

In [ ]:
# Entraînement + prévisions 2025-2030
forecasts = {}
for lib in top_indicateurs:
    df_ind = prep_prophet(df_dep, lib)
    m = Prophet(changepoint_prior_scale=0.05,
                yearly_seasonality=False, weekly_seasonality=False)
    m.fit(df_ind)
    # Les données vont jusqu'à 2021 → prévoir 9 ans pour atteindre 2030
    future = m.make_future_dataframe(periods=9, freq='YS')
    forecasts[lib] = m.predict(future)
    print(f'OK {lib[:50]}')

In [ ]:
# Backtesting : train 2012-2018, test 2019-2021
backtest_results = []
print('Backtest (train 2012-2018, test 2019-2021)')
for lib in top_indicateurs:
    df_bt = prep_prophet(df_dep, lib)
    train = df_bt[df_bt['ds'].dt.year <= 2018]
    test  = df_bt[df_bt['ds'].dt.year >= 2019]
    if len(train) < 4 or len(test) == 0: continue
    m = Prophet(changepoint_prior_scale=0.05,
                yearly_seasonality=False, weekly_seasonality=False)
    m.fit(train)
    fc = m.predict(m.make_future_dataframe(periods=len(test), freq='YS'))
    rmse = rmse_safe(test['y'].values, fc['yhat'].iloc[-len(test):].values)
    backtest_results.append({'Crime': lib[:45], 'RMSE_Prophet': rmse})
    print(f'  {lib[:45]:<48} RMSE: {rmse:,.0f}')

df_backtest = pd.DataFrame(backtest_results)
print(f'\nRMSE moyen : {df_backtest["RMSE_Prophet"].mean():,.0f}')

In [ ]:
# Visualisation prévisions
fig, axes = plt.subplots(len(top_indicateurs), 1, figsize=(13, 6 * len(top_indicateurs)))
if len(top_indicateurs) == 1: axes = [axes]

for ax, lib in zip(axes, top_indicateurs):
    hist = prep_prophet(df_dep, lib)
    fc   = forecasts[lib]
    last = hist['ds'].max()

    ax.plot(hist['ds'], hist['y'], 'o-', color='darkblue', lw=2.5, ms=8, label='Historique')
    fut = fc[fc['ds'] > last]
    ax.plot(fut['ds'], fut['yhat'], '--', color='red', lw=2.5, label='Prévision')
    ax.fill_between(fut['ds'], fut['yhat_lower'], fut['yhat_upper'],
                    alpha=0.2, color='red', label='IC 80%')
    ax.axvline(last, color='gray', ls=':', alpha=0.7)
    ax.set_title(lib[:60], fontweight='bold')
    ax.set_ylabel('Nb faits (France)'); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

plt.suptitle('Prévisions Prophet 2022-2030 — niveau national (somme depts)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('prophet_previsions.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. XGBoost + LightGBM

In [ ]:
import xgboost as xgb
import lightgbm as lgb

def build_features(df):
    """Feature engineering sur df_dep."""
    d = df[['annee', 'departement', 'code_index', 'libelle_crime', 'nb_faits']].copy()
    d['year_sin']   = np.sin(2 * np.pi * d['annee'] / 10)
    d['year_cos']   = np.cos(2 * np.pi * d['annee'] / 10)
    d['year_trend'] = (d['annee'] - d['annee'].min()) / max(1, d['annee'].max() - d['annee'].min())
    d['lag1']       = d['nb_faits'].shift(1).fillna(d['nb_faits'].mean())
    d['lag2']       = d['nb_faits'].shift(2).fillna(d['nb_faits'].mean())
    d['roll3']      = d['nb_faits'].rolling(3, min_periods=1).mean()
    d['dept_mean']  = d.groupby('departement')['nb_faits'].transform('mean')
    d['crime_mean'] = d.groupby('code_index')['nb_faits'].transform('mean')
    d['dept_code']  = pd.factorize(d['departement'])[0]
    d['crime_code'] = pd.factorize(d['code_index'])[0]
    return d.replace([np.inf, -np.inf], 0).fillna(0)

FEATURE_COLS = ['year_sin', 'year_cos', 'year_trend',
                'lag1', 'lag2', 'roll3',
                'dept_mean', 'crime_mean', 'dept_code', 'crime_code']

df_feat = build_features(df_dep)
X = df_feat[FEATURE_COLS].copy()
y = df_feat['nb_faits'].copy()
print(f'Features : {X.shape} | Toutes finies : {np.isfinite(X).all().all()}')

In [ ]:
tscv = TimeSeriesSplit(n_splits=3)
xgb_cv, lgb_cv = [], []

print('Cross-validation XGBoost + LightGBM')
for fold, (tr, te) in enumerate(tscv.split(X)):
    mx = xgb.XGBRegressor(n_estimators=100, max_depth=4, learning_rate=0.1,
                           random_state=42+fold, verbosity=0, n_jobs=1)
    mx.fit(X.iloc[tr], y.iloc[tr])
    rx = np.sqrt(mean_squared_error(y.iloc[te], mx.predict(X.iloc[te])))
    xgb_cv.append(rx)

    ml = lgb.LGBMRegressor(n_estimators=150, max_depth=4, learning_rate=0.1,
                            num_leaves=31, random_state=42+fold, verbose=-1, n_jobs=1)
    ml.fit(X.iloc[tr], y.iloc[tr])
    rl = np.sqrt(mean_squared_error(y.iloc[te], ml.predict(X.iloc[te])))
    lgb_cv.append(rl)

    print(f'  Fold {fold+1} -> XGB: {rx:,.0f} | LGB: {rl:,.0f}')

print(f'\nXGBoost  CV : {np.mean(xgb_cv):,.0f} +/- {np.std(xgb_cv):,.0f}')
print(f'LightGBM CV : {np.mean(lgb_cv):,.0f} +/- {np.std(lgb_cv):,.0f}')

In [ ]:
# Modèles finaux
xgb_final = xgb.XGBRegressor(n_estimators=200, max_depth=4, learning_rate=0.1,
                               subsample=0.8, colsample_bytree=0.8,
                               random_state=42, verbosity=0, n_jobs=1)
xgb_final.fit(X, y)

lgb_final = lgb.LGBMRegressor(n_estimators=300, max_depth=4, learning_rate=0.08,
                                num_leaves=50, random_state=42, verbose=-1, n_jobs=1)
lgb_final.fit(X, y)
print('Modèles finaux OK')

# Feature importance
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
for ax, model, title, palette in [
    (ax1, xgb_final, 'XGBoost', 'rocket'),
    (ax2, lgb_final, 'LightGBM', 'viridis')
]:
    imp = pd.DataFrame({'feature': FEATURE_COLS,
                        'importance': model.feature_importances_})\
            .sort_values('importance', ascending=False)
    sns.barplot(data=imp, y='feature', x='importance', palette=palette, ax=ax)
    ax.set_title(f'{title} — Feature Importance', fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# Prévisions 2030 par crime (top 5)
xgb_2030, lgb_2030 = {}, {}

for lib in top_indicateurs:
    code = df_dep[df_dep['libelle_crime'] == lib]['code_index'].iloc[0]
    mask = (df_feat['libelle_crime'] == lib)
    if mask.sum() == 0: continue
    row = X[mask].iloc[-1].copy()
    row['year_sin']   = np.sin(2 * np.pi * 2030 / 10)
    row['year_cos']   = np.cos(2 * np.pi * 2030 / 10)
    row['year_trend'] = 1.0
    v = row.values.reshape(1, -1)
    xgb_2030[lib] = float(xgb_final.predict(v)[0])
    lgb_2030[lib] = float(lgb_final.predict(v)[0])

print('Prévisions 2030 (nb faits, niveau national) :')
for lib in top_indicateurs:
    print(f'  {lib[:45]:<48} XGB: {xgb_2030.get(lib,0):>10,.0f} | LGB: {lgb_2030.get(lib,0):>10,.0f}')

## 8. Rapport Final

In [ ]:
# Tableau comparatif RMSE
rmse_cmp = pd.DataFrame({
    'Modele': ['Prophet', 'XGBoost', 'LightGBM'],
    'RMSE_CV': [df_backtest['RMSE_Prophet'].mean(),
                np.mean(xgb_cv), np.mean(lgb_cv)]
}).sort_values('RMSE_CV')
print('Comparaison modèles :')
print(rmse_cmp.round(0).to_string(index=False))

In [ ]:
# Synthèse prévisions 2030
trio = []
for lib in top_indicateurs:
    pred_l = lgb_2030.get(lib, np.nan)
    pred_x = xgb_2030.get(lib, np.nan)
    fc = forecasts.get(lib)
    pred_p = fc[fc['ds'].dt.year == 2030]['yhat'].values[0] \
             if fc is not None and (fc['ds'].dt.year == 2030).any() else np.nan
    ens = 0.4*pred_l + 0.4*pred_x + 0.2*pred_p \
          if not any(np.isnan([pred_l, pred_x, pred_p])) else np.nan
    trio.append({'Crime': lib[:40],
                 'LightGBM': round(pred_l), 'XGBoost': round(pred_x),
                 'Prophet': round(pred_p) if not np.isnan(pred_p) else np.nan,
                 'Ensemble': round(ens) if not np.isnan(ens) else np.nan})

df_trio = pd.DataFrame(trio)
print('Prévisions 2030 (nb faits France, PN+GN) :')
print(df_trio.to_string(index=False))

In [ ]:
# Export Excel multi-onglets
with pd.ExcelWriter('rapport_delinquance_dep_2030.xlsx', engine='openpyxl') as w:
    df_dep.to_excel(w, sheet_name='Données_dept', index=False)
    df_dep_svc.to_excel(w, sheet_name='Données_dept_service', index=False)
    df_trio.to_excel(w, sheet_name='Previsions_2030', index=False)
    df_backtest.to_excel(w, sheet_name='Backtest_Prophet', index=False)
    rmse_cmp.to_excel(w, sheet_name='RMSE_Comparaison', index=False)
    top20_dep.reset_index(name='Total_faits').to_excel(
        w, sheet_name='Top20_Departements', index=False)

print('Exporté : rapport_delinquance_dep_2030.xlsx')
print('\n' + '='*65)
print('RAPPORT FINAL — DÉLINQUANCE FRANCE PAR DÉPARTEMENT 2012-2021')
print('='*65)
print(f'Onglets analysés : {len(onglets)}')
print(f'Départements     : {df_dep["departement"].nunique()}')
print(f'Index de crimes  : {df_dep["code_index"].nunique()}')
print(f'Total faits      : {df_dep["nb_faits"].sum():,.0f}')
print()
print('RMSE cross-validation :')
for _, row in rmse_cmp.iterrows():
    print(f'  {row["Modele"]:<12}: {row["RMSE_CV"]:,.0f}')
print('='*65)